# Tarea: Construcción del Dataset Integrado (Demografía, Salud y Economía)

### Objetivo de la Tarea:
Completar la construcción del dataset agregando las variables **`percentage_health`** y **`gdp`** al dataset demográfico creado en clase.

La salida final debe contener **únicamente las 8 columnas requeridas**:
1. `entity`
2. `year`
3. `population`
4. `births`
5. `deaths`
6. `child_deaths_under_5`
7. `percentage_health`
8. `gdp`

In [ ]:
import pandas as pd
import numpy as np

# Configuración de headers para Our World in Data
headers = {'User-Agent': 'Our World In Data data fetch/1.0'}

## PARTE 1. Dataset Base Creado en Clase (Demografía UN WPP)

Cargamos y estructuramos las 6 variables base:
* `entity` y `year` (País y Año)
* `population` (Población total estimada)
* `births` (Nacimientos anuales)
* `deaths` (Defunciones anuales)
* `child_deaths_under_5` (Muertes de niños menores a 5 años)

In [ ]:
# 1. Nacimientos y Defunciones
url_bd = "https://ourworldindata.org/grapher/births-and-deaths-projected-to-2100.csv?v=1&csvType=full&useColumnShortNames=true"
df_bd = pd.read_csv(url_bd, storage_options=headers)
df_bd = df_bd.rename(columns={
    'births__sex_all__age_all__variant_estimates': 'births',
    'deaths__sex_all__age_all__variant_estimates': 'deaths'
})[['entity', 'code', 'year', 'births', 'deaths']]

# 2. Población
url_pop = "https://ourworldindata.org/grapher/population-with-un-projections.csv?v=1&csvType=full&useColumnShortNames=true"
df_pop = pd.read_csv(url_pop, storage_options=headers)
df_pop = df_pop.rename(columns={
    'population__sex_all__age_all__variant_estimates': 'population'
})[['entity', 'code', 'year', 'population']]

# 3. Muertes en menores de 5 años (child_deaths_under_5)
url_cd = "https://ourworldindata.org/grapher/number-of-child-deaths-unwpp.csv?v=1&csvType=full&useColumnShortNames=true"
df_cd = pd.read_csv(url_cd, storage_options=headers)
df_cd = df_cd.rename(columns={
    'deaths__sex_all__age_0_4__variant_estimates': 'child_deaths_under_5'
})[['entity', 'code', 'year', 'child_deaths_under_5']]

# Unir el dataset base de la clase
df_clase = df_pop.merge(df_bd, on=['entity', 'code', 'year'], how='inner')
df_clase = df_clase.merge(df_cd, on=['entity', 'code', 'year'], how='inner')

print("Dimensiones del dataset de clase:", df_clase.shape)
df_clase.head(3)

Dimensiones del dataset de clase: (18942, 7)


,entity,code,year,population,births,deaths,child_deaths_under_5
0,Afghanistan,AFG,1950,7776180.0,383985.0,290972.0,160791
1,Afghanistan,AFG,1951,7879343.0,391002.0,288752.0,158566
2,Afghanistan,AFG,1952,7987784.0,397663.0,288059.0,157832


## PARTE 2. Extracción de los Nuevos Indicadores Solicitados

### 2.1 Indicador: Total healthcare spending as a share of GDP (2000 to 2023)
Extraemos la variable `percentage_health`.

In [ ]:
# Carga de Total healthcare spending as a share of GDP
url_health = "https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv?v=1&csvType=full&useColumnShortNames=true"
df_health = pd.read_csv(url_health, storage_options=headers)

# Renombrar la columna a 'percentage_health' como lo pide la tarea
df_health = df_health.rename(columns={
    'current_health_expenditure__che__as_percentage_of_gross_domestic_product__gdp__pct': 'percentage_health'
})

print("df_health shape:", df_health.shape)
df_health[['entity', 'year', 'percentage_health']].head(3)

df_health shape: (4756, 4)


,entity,year,percentage_health
0,Afghanistan,2002,9.443391
1,Afghanistan,2003,8.941258
2,Afghanistan,2004,9.808474


### 2.2 Indicador: GDP per capita
Extraemos la variable `gdp`.

In [ ]:
# Carga de GDP per capita
url_gdp = "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true"
df_gdp = pd.read_csv(url_gdp, storage_options=headers)

# Renombrar la columna a 'gdp'
df_gdp = df_gdp.rename(columns={
    'ny_gdp_pcap_pp_kd': 'gdp'
})

print("df_gdp shape:", df_gdp.shape)
df_gdp[['entity', 'year', 'gdp']].head(3)

df_gdp shape: (7445, 5)


,entity,year,gdp
0,Afghanistan,2000,1617.8264
1,Afghanistan,2001,1454.1108
2,Afghanistan,2002,1774.3087


## PARTE 3. Construcción del Dataset Final con las 8 Columnas

Unimos las variables `percentage_health` y `gdp` al dataset creado en clase mediante `pd.merge` por `entity` y `year`.

In [ ]:
# Merge con df_health y df_gdp
df_completo = df_clase.merge(
    df_health[['entity', 'year', 'percentage_health']],
    on=['entity', 'year'],
    how='inner'
).merge(
    df_gdp[['entity', 'year', 'gdp']],
    on=['entity', 'year'],
    how='inner'
)

# Definir la lista exacta de las 8 columnas requeridas por el profesor
columnas_salida = [
    'entity', 
    'year', 
    'population', 
    'births', 
    'deaths', 
    'child_deaths_under_5', 
    'percentage_health', 
    'gdp'
]

# Filtrar y ordenar las columnas requeridas
df_final = df_completo[columnas_salida].copy()

# Ordenar por país y año
df_final = df_final.sort_values(by=['entity', 'year']).reset_index(drop=True)

print("¡Dataset completado con éxito!")
print(f"Dimensiones finales: {df_final.shape[0]} filas y {df_final.shape[1]} columnas")
print("\nLista de columnas:")
for i, col in enumerate(df_final.columns, 1):
    print(f"{i}. {col}")

¡Dataset completado con éxito!
Dimensiones finales: 4428 filas y 8 columnas

Lista de columnas:
1. entity
2. year
3. population
4. births
5. deaths
6. child_deaths_under_5
7. percentage_health
8. gdp


## PARTE 4. Inspección del Dataset Final

In [ ]:
# Primeras filas del dataset final
df_final.head(10)

,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
0,Afghanistan,2002,21378123.0,1015202.0,239161.0,123215,9.443391,1774.3087
1,Afghanistan,2003,22733053.0,1069727.0,243573.0,123894,8.941258,1815.9282
2,Afghanistan,2004,23560656.0,1095372.0,244435.0,122315,9.808474,1776.9182
3,Afghanistan,2005,24404575.0,1097038.0,241932.0,118420,9.948289,1908.1147
4,Afghanistan,2006,25424100.0,1120880.0,245586.0,116622,10.622766,1929.7239
5,Afghanistan,2007,25909852.0,1135656.0,245809.0,112800,9.904675,2155.3530
6,Afghanistan,2008,26482631.0,1126525.0,235284.0,106511,10.256495,2191.5044
7,Afghanistan,2009,27466102.0,1158017.0,237450.0,106480,9.818487,2565.0220
8,Afghanistan,2010,28284088.0,1181113.0,237640.0,104547,8.569672,2848.5862
9,Afghanistan,2011,29347708.0,1202456.0,235724.0,101679,8.561908,2757.0525


In [ ]:
# Últimas filas del dataset final
df_final.tail(10)

,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
4418,Zimbabwe,2014,14207367.0,503038.0,136688.0,32501,8.133513,5081.1343
4419,Zimbabwe,2015,14399008.0,488386.0,132188.0,30095,7.452041,5102.7363
4420,Zimbabwe,2016,14600297.0,477484.0,128714.0,28013,7.258456,5070.4243
4421,Zimbabwe,2017,14812484.0,471228.0,127191.0,26491,6.174489,5234.4067
4422,Zimbabwe,2018,15034457.0,470991.0,125092.0,25211,4.673280,5415.4930
4423,Zimbabwe,2019,15271377.0,475267.0,126468.0,24647,3.232681,4993.8650
4424,Zimbabwe,2020,15526887.0,481152.0,126365.0,23828,2.954401,4527.7393
4425,Zimbabwe,2021,15797220.0,488642.0,138738.0,23391,2.671119,4827.1094
4426,Zimbabwe,2022,16069061.0,496240.0,124995.0,23249,3.395195,5036.7830
4427,Zimbabwe,2023,16340829.0,496917.0,124412.0,22505,2.926016,5218.0454


In [ ]:
# Tipos de datos e información general
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4428 entries, 0 to 4427
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   entity                4428 non-null   str    
 1   year                  4428 non-null   int64  
 2   population            4428 non-null   float64
 3   births                4428 non-null   float64
 4   deaths                4428 non-null   float64
 5   child_deaths_under_5  4428 non-null   int64  
 6   percentage_health     4428 non-null   float64
 7   gdp                   4428 non-null   float64
dtypes: float64(5), int64(2), str(1)
memory usage: 276.9 KB


### Ejemplo: Consulta para México (2000 a 2023)

In [ ]:
# Visualización de los datos correspondientes a México
df_mexico = df_final[df_final['entity'] == 'Mexico']
df_mexico

,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
2545,Mexico,2000,98625553.0,2385977.0,540567.0,66928,4.244365,20470.494
2546,Mexico,2001,100099101.0,2380261.0,543803.0,64012,4.582014,20078.219
2547,Mexico,2002,101548627.0,2374140.0,553391.0,61210,4.830428,19744.793
2548,Mexico,2003,102978517.0,2351177.0,563140.0,58280,5.539788,19701.463
2549,Mexico,2004,104394131.0,2335226.0,575925.0,55625,5.683502,20127.225
2550,Mexico,2005,105811504.0,2318498.0,585859.0,53204,5.581101,20277.256
2551,Mexico,2006,107253666.0,2303997.0,597892.0,50857,5.406479,20965.826
2552,Mexico,2007,108774356.0,2296268.0,611318.0,48847,5.506680,21102.270
2553,Mexico,2008,110374286.0,2297950.0,627177.0,47137,5.445874,20992.560
2554,Mexico,2009,111999720.0,2303532.0,642292.0,45610,5.847865,19385.543


## PARTE 5. Exportar el Dataset Resultante

Guardamos el dataset final como archivo CSV para entrega o uso posterior.

In [ ]:
# Guardar el dataset completado en un archivo CSV
nombre_archivo_csv = "dataset_health_gdp_demografia_final.csv"
df_final.to_csv(nombre_archivo_csv, index=False)
print(f"Archivo guardado exitosamente: {nombre_archivo_csv}")

Archivo guardado exitosamente: dataset_health_gdp_demografia_final.csv